# Field Shift — risco climático de plantio com dados da NASA (protótipo)

Este notebook:
1. Baixa a série **diária** do **NASA POWER** (chuva, temperatura máx./mín.) para um ponto (padrão: Uberlândia-MG).
2. Calcula a evapotranspiração de referência (Hargreaves) e roda um **balanço hídrico simplificado tipo "balde"**, inspirado no modelo SARRA usado pelo ZARC.
3. Para cada um dos **36 decêndios de semeadura** e **6 classes de água disponível no solo (AD1–AD6)**, calcula o **ISNA** (ETr/ETm) por fase da soja e o **risco** = frequência de anos em que o ISNA ficou abaixo do limite crítico (0,50 na fase inicial; 0,55 na reprodutiva).
4. **Field Shift:** compara o risco entre dois períodos (1991–2008 × 2009–2025) para mostrar quais janelas de plantio ficaram mais arriscadas.

⚠️ Os parâmetros da cultura (duração das fases, Kc) e as classes AD5/AD6 são **provisórios**: calibrar com a documentação do ZARC e validar contra a tábua oficial de risco.

In [ ]:
# ===== Parâmetros do local e do período =====
MUNICIPIO = "Uberlândia-MG"
LAT, LON = -18.9186, -48.2772
ANO_INI, ANO_FIM = 1991, 2025
PERIODO_1 = range(1991, 2009)   # 1991–2008
PERIODO_2 = range(2009, 2025)   # 2009–2024 (safras com ciclo completo nos dados)

In [ ]:
# ===== Download NASA POWER (diário, comunidade AG) =====
import requests, pandas as pd, numpy as np

url = "https://power.larc.nasa.gov/api/temporal/daily/point"
params = {
    "parameters": "PRECTOTCORR,T2M_MAX,T2M_MIN",
    "community": "AG",
    "latitude": LAT, "longitude": LON,
    "start": f"{ANO_INI}0101", "end": f"{ANO_FIM}1231",
    "format": "JSON",
}
r = requests.get(url, params=params, timeout=120)
r.raise_for_status()
p = r.json()["properties"]["parameter"]

df = pd.DataFrame({
    "chuva": pd.Series(p["PRECTOTCORR"]),
    "tmax": pd.Series(p["T2M_MAX"]),
    "tmin": pd.Series(p["T2M_MIN"]),
})
df.index = pd.to_datetime(df.index, format="%Y%m%d")
df = df.replace(-999, np.nan)
print("Dias faltantes:\n", df.isna().sum())
df = df.interpolate(limit=5).fillna({"chuva": 0})
df.describe()

In [ ]:
# ===== Funções do modelo (balanço hídrico, ISNA e risco) =====
import numpy as np
import pandas as pd

# ---------------- Parâmetros (ajustáveis / calibrar com a documentação do ZARC) ----------------
# Água disponível por classe (mm por cm de solo). AD1–AD4: pontos médios das faixas da Embrapa Soja.
# AD5 e AD6: valores provisórios — confirmar na documentação do SISZARC.
CLASSES_AD = {"AD1": 0.40, "AD2": 0.535, "AD3": 0.705, "AD4": 0.93, "AD5": 1.20, "AD6": 1.50}
PROF_RAIZ_CM = 60  # profundidade efetiva usada no ZARC soja

# Soja de ciclo médio (~120 dias). Duração das fases e Kc são provisórios.
FASES_SOJA = [  # (nome, dias, Kc)
    ("F1_inicial", 20, 0.40),
    ("F2_vegetativo", 30, 0.80),
    ("F3_reprodutivo", 45, 1.15),
    ("F4_maturacao", 25, 0.60),
]
LIMITE_ISNA = {"F1_inicial": 0.50, "F3_reprodutivo": 0.55}  # cortes usados no ZARC soja
P_DEPLECAO = 0.5  # fração da CAD consumida sem estresse


def radiacao_extraterrestre(lat_graus, doy):
    """Ra em mm/dia equivalentes (FAO-56)."""
    phi = np.radians(lat_graus)
    dr = 1 + 0.033 * np.cos(2 * np.pi * doy / 365)
    delta = 0.409 * np.sin(2 * np.pi * doy / 365 - 1.39)
    ws = np.arccos(np.clip(-np.tan(phi) * np.tan(delta), -1, 1))
    ra_mj = (24 * 60 / np.pi) * 0.0820 * dr * (
        ws * np.sin(phi) * np.sin(delta) + np.cos(phi) * np.cos(delta) * np.sin(ws))
    return 0.408 * ra_mj


def eto_hargreaves(tmax, tmin, lat, doy):
    tmed = (tmax + tmin) / 2
    ra = radiacao_extraterrestre(lat, doy)
    return np.maximum(0.0023 * (tmed + 17.8) * np.sqrt(np.maximum(tmax - tmin, 0)) * ra, 0)


def _passo_balde(S, chuva, etm, cad):
    S = min(cad, S + chuva)
    limiar = (1 - P_DEPLECAO) * cad
    ks = 1.0 if S >= limiar else S / limiar
    etr = min(etm * ks, S)
    return S - etr, etr


def isna_por_fase(chuva, eto, i0, cad, fases=FASES_SOJA, spinup=30, kc_pousio=0.3):
    """Simula o balanço hídrico a partir do índice i0 (semeadura). Retorna ISNA por fase ou None."""
    n_ciclo = sum(d for _, d, _ in fases)
    if i0 - spinup < 0 or i0 + n_ciclo > len(chuva):
        return None
    S = 0.5 * cad
    for j in range(i0 - spinup, i0):  # aquecimento com solo em pousio
        S, _ = _passo_balde(S, chuva[j], kc_pousio * eto[j], cad)
    out, j = {}, i0
    for nome, dias, kc in fases:
        s_etr = s_etm = 0.0
        for _ in range(dias):
            etm = kc * eto[j]
            S, etr = _passo_balde(S, chuva[j], etm, cad)
            s_etr += etr
            s_etm += etm
            j += 1
        out[nome] = s_etr / s_etm if s_etm > 0 else 1.0
    return out


def datas_decendios(ano):
    return [pd.Timestamp(ano, m, d) for m in range(1, 13) for d in (1, 11, 21)]


def simular_risco(df, lat, anos):
    """df: índice diário (DatetimeIndex) com colunas chuva, tmax, tmin.
    Retorna uma linha por ano × decêndio × classe de solo com o ISNA e se houve falha."""
    df = df.sort_index().copy()
    df["eto"] = eto_hargreaves(df.tmax.values, df.tmin.values, lat, df.index.dayofyear.values)
    chuva, eto = df.chuva.values, df.eto.values
    pos = {d: i for i, d in enumerate(df.index)}
    linhas = []
    for classe, ad in CLASSES_AD.items():
        cad = ad * PROF_RAIZ_CM
        for ano in anos:
            for k, data in enumerate(datas_decendios(ano), start=1):
                i0 = pos.get(data)
                if i0 is None:
                    continue
                r = isna_por_fase(chuva, eto, i0, cad)
                if r is None:
                    continue
                falha = any(r[f] < lim for f, lim in LIMITE_ISNA.items())
                linhas.append({"classe_solo": classe, "ano": ano, "decendio": k,
                               "data_semeadura": data.date(), **{f"isna_{f}": v for f, v in r.items()},
                               "falha": falha})
    return pd.DataFrame(linhas)


def classificar(risco):
    if risco <= 0.20:
        return "20%"
    if risco <= 0.30:
        return "30%"
    if risco <= 0.40:
        return "40%"
    return "não indicado"


def tabua_risco(sim, anos=None):
    s = sim if anos is None else sim[sim.ano.isin(anos)]
    t = s.groupby(["classe_solo", "decendio"], as_index=False).agg(risco=("falha", "mean"), n_anos=("ano", "nunique"))
    t["faixa"] = t.risco.apply(classificar)
    return t

In [ ]:
# ===== Simulação para todos os anos =====
sim = simular_risco(df, LAT, range(ANO_INI, ANO_FIM))
tabua = tabua_risco(sim)
tabua.to_csv("tabua_risco_nasa_power.csv", index=False)
print(sim.shape)
tabua.pivot(index="decendio", columns="classe_solo", values="faixa")

In [ ]:
# ===== Field Shift: o risco mudou entre os dois períodos? =====
import matplotlib.pyplot as plt

t1 = tabua_risco(sim, PERIODO_1).assign(periodo="1991–2008")
t2 = tabua_risco(sim, PERIODO_2).assign(periodo="2009–2024")
comp = t1.merge(t2, on=["classe_solo", "decendio"], suffixes=("_p1", "_p2"))
comp["variacao_pp"] = (comp.risco_p2 - comp.risco_p1) * 100
comp.to_csv("comparacao_periodos.csv", index=False)

meses = ["Jan","Fev","Mar","Abr","Mai","Jun","Jul","Ago","Set","Out","Nov","Dez"]
rotulos = [f"{m}-{d}" for m in meses for d in ("1","2","3")]

classe = "AD3"
c = comp[comp.classe_solo == classe]
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(c.decendio, c.risco_p1 * 100, label="1991–2008")
ax.plot(c.decendio, c.risco_p2 * 100, label="2009–2024")
for lim in (20, 30, 40):
    ax.axhline(lim, ls=":", lw=0.8, color="gray")
ax.set_xticks(range(1, 37), rotulos, rotation=90, fontsize=7)
ax.set_ylabel("Risco de falha (%)")
ax.set_title(f"Soja (ciclo médio) — {MUNICIPIO} — solo {classe}: risco por decêndio de semeadura")
ax.legend(); plt.tight_layout(); plt.show()

piv = comp.pivot(index="classe_solo", columns="decendio", values="variacao_pp")
fig, ax = plt.subplots(figsize=(12, 3))
im = ax.imshow(piv.values, cmap="RdBu_r", vmin=-30, vmax=30, aspect="auto")
ax.set_yticks(range(len(piv.index)), piv.index)
ax.set_xticks(range(36), rotulos, rotation=90, fontsize=7)
fig.colorbar(im, label="Variação do risco (p.p.)")
ax.set_title("Mudança no risco entre períodos (vermelho = ficou mais arriscado)")
plt.tight_layout(); plt.show()

In [ ]:
# ===== Tendência da chuva na estação (out–mar) por safra =====
from scipy.stats import kendalltau

d = df.copy()
d["safra"] = np.where(d.index.month >= 7, d.index.year, d.index.year - 1)
est = d[d.index.month.isin([10, 11, 12, 1, 2, 3])].groupby("safra").chuva.sum()
est = est.loc[ANO_INI:ANO_FIM - 1]
tau, pval = kendalltau(est.index, est.values)
incl = np.polyfit(est.index, est.values, 1)[0]
print(f"Chuva out–mar: tendência {incl:+.1f} mm/ano | Kendall tau = {tau:.2f}, p = {pval:.3f}")
est.plot(marker="o", figsize=(10, 3), title="Chuva acumulada out–mar por safra (NASA POWER)", ylabel="mm");

## Próximos passos
- **Calibrar** fases e Kc da soja (e criar milho safrinha, sorgo, feijão) com a documentação do ZARC.
- **Validar:** comparar `tabua_risco_nasa_power.csv` com a tábua oficial do ZARC para o município (concordância das faixas por decêndio).
- **Escalar:** rodar para vários municípios (lista de lat/lon) e montar o mapa.
- **Rotação:** usar as janelas de baixo risco de cada cultura como entrada do motor de rotação (soja → milho safrinha → cobertura).